In [1]:
import json
import os
import re

import data_utils
from concept_generation_qwen import LocalQwenGenerator

/home/ensta/ensta-gassem/Label-free-CBM-Audio/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


In [17]:
dataset = "esc50"
prompt_type = "around"  # important | superclass | around

model_id = "Qwen/Qwen3.5-9B"
device = "cuda"

num_trials = 2
max_new_tokens = 256
temperature = 0.7
top_p = 0.9

In [18]:
prompts = {
    "important": """You are an expert in audio perception and sound classification.

Task: List around 5 of the most important *auditory cues* that help distinguish a sound as belonging to the class "{class}".

Requirements:
- Only include features that can be directly heard (no visual or semantic descriptions)
- Focus on discriminative cues (what makes this class distinct from others)
- Use short noun phrases (2-4 words)
- Avoid redundancy
- Avoid overly generic terms (e.g., "noise", "sound")
- Prefer mid-level concepts (e.g., "rhythmic barking", not just "bark", but not overly detailed)

Examples:

Class: dog
- rhythmic barking
- low growling
- sharp yelping
- panting breaths
- claw tapping

Class: rain
- continuous water drops
- irregular splashing
- soft surface tapping
- distant thunder rumble
- wind-driven rain

Now list around 5 of the most important auditory cues for the class "{class}":
""",
    "superclass": """You are an expert in sound taxonomy.

Task: Provide superclass categories for the sound class "{class}".

Requirements:
- Return 3-6 categories
- Include multiple levels of abstraction (specific → general)
- Each item should be a short noun phrase
- Categories must describe *types of sounds*, not sources or contexts
- Avoid redundancy

Examples:

Class: dog
- animal vocalization
- mammal sound
- domestic animal sound
- biological sound

Class: rain
- weather sound
- natural environmental sound
- atmospheric sound
- ambient soundscape

Now provide superclasses for the sound class "{class}":
""",
    "around": """You are analyzing real-world audio recordings.

Task: List around 5 sounds that commonly co-occur in recordings where the main sound class is "{class}".

Requirements:
- Include only *other sounds*, not the main class itself
- Focus on realistic acoustic environments
- Use short noun phrases (2-4 words)
- Avoid redundancy
- Prefer concrete, audible elements

Examples:

Class: dog
- human speech
- footsteps on ground
- leash jingle
- door opening
- park ambience

Class: rain
- wind noise
- distant traffic
- thunder rumble
- umbrella movement
- flowing water

Now list around 5 sounds commonly heard around a "{class}" recording:
"""  }

base_prompt = prompts[prompt_type]
base_prompt[:300]

'You are analyzing real-world audio recordings.\n\nTask: List around 5 sounds that commonly co-occur in recordings where the main sound class is "{class}".\n\nRequirements:\n- Include only *other sounds*, not the main class itself\n- Focus on realistic acoustic environments\n- Use short noun phrases (2-4 wo'

In [19]:
cls_file = data_utils.LABEL_FILES[dataset]
with open(cls_file, "r", encoding="utf-8") as f:
    classes = [line.strip() for line in f if len(line.strip()) > 0]

save_dir = "data/concept_sets/qwen_init"
os.makedirs(save_dir, exist_ok=True)
save_path = "{}/qwen_{}_{}.json".format(save_dir, dataset, prompt_type)

len(classes), classes[:10], save_path

(50,
 ['dog',
  'rooster',
  'pig',
  'cow',
  'frog',
  'cat',
  'hen',
  'insects',
  'sheep',
  'crow'],
 'data/concept_sets/qwen_init/qwen_esc50_around.json')

In [5]:
generator = LocalQwenGenerator(model_id=model_id, device=device)
"generator ready"

'generator ready'

In [6]:
generator.generate(
    base_prompt.format(**{"class": "Dinosaur"}),
    max_new_tokens=32,
    temperature=0.2,
    top_p=0.9,
    enable_thinking=False,
 )

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

'- Low-frequency roars\n- Deep guttural grunts\n- Heavy foot stomps\n- Tail swishing rustles\n- Sudden hissing'

In [20]:
feature_dict = {}
meta_terms = ["analyze", "constraint", "task", "goal", "thinking", "reasoning", "instructions", "output format"]

for i, label in enumerate(classes):
    feature_dict[label] = set()
    print("\n", i, label)
    for _ in range(num_trials):
        try:
            formatted_prompt = base_prompt.format(**{"class": label})
        except (KeyError, IndexError):
            formatted_prompt = base_prompt.format(label)

        response = generator.generate(
            formatted_prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            enable_thinking=False,
        )

        features = response.split("\n")
        cleaned = []
        for feat in features:
            feat = re.sub(r"^[-*\\d\\.)\\s]+", "", feat).strip()
            feat = feat.strip('"').strip("'").strip().strip(",")
            if len(feat) == 0:
                continue

            lower_feat = feat.lower()
            if any(term in lower_feat for term in meta_terms):
                continue
            if len(feat.split()) > 6:
                continue

            cleaned.append(feat)

        feature_dict[label].update(cleaned)

    feature_dict[label] = sorted(list(feature_dict[label]))

len(feature_dict), len(next(iter(feature_dict.values()))) if len(feature_dict) > 0 else 0


 0 dog

 1 rooster



 2 pig

 3 cow

 4 frog

 5 cat

 6 hen

 7 insects

 8 sheep

 9 crow

 10 rain

 11 sea_waves

 12 crackling_fire

 13 crickets

 14 chirping_birds

 15 water_drops

 16 wind

 17 pouring_water

 18 toilet_flush

 19 thunderstorm

 20 crying_baby

 21 sneezing

 22 clapping

 23 breathing

 24 coughing

 25 footsteps

 26 laughing

 27 brushing_teeth

 28 snoring

 29 drinking_sipping

 30 door_wood_knock

 31 mouse_click

 32 keyboard_typing

 33 door_wood_creaks

 34 can_opening

 35 washing_machine

 36 vacuum_cleaner

 37 clock_alarm

 38 clock_tick

 39 glass_breaking

 40 helicopter

 41 chainsaw

 42 siren

 43 car_horn

 44 engine

 45 train

 46 church_bells

 47 airplane

 48 fireworks

 49 hand_saw


(50, 8)

In [21]:
with open(save_path, "w", encoding="utf-8") as outfile:
    json.dump(feature_dict, outfile, indent=4, ensure_ascii=True)

save_path

'data/concept_sets/qwen_init/qwen_esc50_around.json'

In [22]:
with open(save_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

example_label = classes[1]
example_label, saved[example_label], len(saved)

('rooster',
 ['chicken clucking',
  'crowing neighbors',
  'distant traffic',
  'distant traffic noise',
  'farm machinery',
  'human laughter',
  'human shouting',
  'metal gate creaking',
  'wind rustling leaves'],
 50)

In [ ]:
"done"